## Niki Naderzad
## Group 3
## Group Assignment Task 2

In [ ]:
import argparse
import os

from pyspark.sql import SparkSession
# Parse command-line arguments
parser = argparse.ArgumentParser()
parser.add_argument(
    "--is-local",
    type=str,
    default="true",
    help="Whether running in local mode",
)
args, unknown = parser.parse_known_args()
is_local = args.is_local.lower() == "true"

print(f"Is local environment: {is_local}")

# Build SparkSession with conditional configuration
if is_local:
    adc_path = os.path.expanduser(
        "~/.config/gcloud/application_default_credentials.json"
    )
    gcs_connector_jar_path = "../gcs-connector-hadoop3-latest.jar"
    if not os.path.exists(gcs_connector_jar_path):
        raise FileNotFoundError(
            f"GCS connector JAR not found at {gcs_connector_jar_path}"
        )

    sparkgcp = (
        SparkSession.builder.appName("LocalGCS")
        .config("spark.driver.host", "localhost")
        .config(
            "spark.jars",
            gcs_connector_jar_path,
        )
        .config(
            "spark.hadoop.google.cloud.auth.service.account.enable", "true"
        )
        .config(
            "spark.hadoop.google.cloud.auth.service.account.json.keyfile",
            adc_path,
        )
        .config(
            "spark.hadoop.fs.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        )
        .config(
            "spark.hadoop.fs.AbstractFileSystem.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS",
        )
        .getOrCreate()
    )
else:
    sparkgcp = SparkSession.builder.appName("GCSCluster").getOrCreate()

sc = sparkgcp.sparkContext
sc.setLogLevel("WARN")

print("Connected to Spark cluster!")
print(f"Spark Version: {sc.version}")
print(f"Master: {sc.master}")
print(f"App ID: {sc.applicationId}")

Is local environment: True
Connected to Spark cluster!
Spark Version: 4.0.1
Master: local[*]
App ID: local-1764633451051


In [47]:
if is_local:
    prefix = "../data/processed/merged"
else:
    prefix = "gs://msds-694-cohort-14-3/data"

num_csv_path = f"{prefix}/num_2020.csv"
pre_csv_path = f"{prefix}/pre_2020.csv"
sub_csv_path = f"{prefix}/sub_2020.csv"
tag_csv_path = f"{prefix}/tag_2020.csv"

print(f"Num CSV path: {num_csv_path}")
print(f"Pre CSV path: {pre_csv_path}")
print(f"Sub CSV path: {sub_csv_path}")
print(f"Tag CSV path: {tag_csv_path}")

num_rdd = sc.textFile(num_csv_path)
pre_rdd = sc.textFile(pre_csv_path)
sub_rdd = sc.textFile(sub_csv_path)
tag_rdd = sc.textFile(tag_csv_path)

# print size of each RDD
print(f"Num RDD size: {num_rdd.count()}")
print(f"Pre RDD size: {pre_rdd.count()}")
print(f"Sub RDD size: {sub_rdd.count()}")
print(f"Tag RDD size: {tag_rdd.count()}")

Num CSV path: ../data/processed/merged/num_2020.csv
Pre CSV path: ../data/processed/merged/pre_2020.csv
Sub CSV path: ../data/processed/merged/sub_2020.csv
Tag CSV path: ../data/processed/merged/tag_2020.csv


Num RDD size: 11493263
Pre RDD size: 2746310
Sub RDD size: 24940
Tag RDD size: 298803


**Do companies with strong operating cashflow also show higher profitability, and does this change for different industries?**

Compute correlation between operating cash flow and net income. Look at how
this relationship differs in various industries to see if there are some of them
that don’t have correlation between profitability and cash flow. Identify
outliers in both profitability and cash flow.


### Create a 5000-line subset from NUM (in memory)

In this cell, I take the first 5,000 lines of the large NUM RDD.  
This satisfies the assignment requirement of creating a smaller “subset” dataset.

Instead of writing this subset out to a local CSV file (which would not be visible to Dataproc),
I keep the subset in memory by turning the list of lines back into an RDD with `sc.parallelize`.
This way the exact same notebook works both on my laptop and on the Dataproc cluster, and I
can still explore the smaller subset in later cells using `num_subset_rdd`.

In [48]:
subset_size = 5000

num_subset_lines = num_rdd.take(subset_size)
print("Subset size (lines taken from num_2020.csv):", len(num_subset_lines))

num_subset_rdd = sc.parallelize(num_subset_lines)
print("num_subset_rdd count:", num_subset_rdd.count())

Subset size (lines taken from num_2020.csv): 5000
num_subset_rdd count: 5000


### Clean and split the subset and SUB files

Here I remove the header row from the subset file and from the SUB file.  
Then I split every line by commas so I can access each column individually.

The NUM file contains numeric financial values like operating cashflow and net income.  
The SUB file contains company-level metadata, including the SIC industry codes.

In [49]:
num_header = num_rdd.first()
sub_header = sub_rdd.first()

num_subset_data = num_subset_rdd.filter(lambda line: line != num_header)
sub_data = sub_rdd.filter(lambda line: line != sub_header)

num_subset_split = num_subset_data.map(lambda line: line.split(","))
sub_split = sub_data.map(lambda line: line.split(","))

print("Example NUM subset row:", num_subset_split.first())
print("Example SUB row:", sub_split.first())

Example NUM subset row: ['0001564590-20-010652', 'AccountsPayableCurrentAndNoncurrent', 'us-gaap/2019', '20181231', '0', 'USD', '', '', '607000.0', '', 'q1', '2020']
Example SUB row: ['0000002178-20-000013', '2178', '"ADAMS RESOURCES & ENERGY', ' INC."', '5172.0', 'US', 'TX', 'HOUSTON', '77027', '17 S. BRIAR HOLLOW LN.', '', '713-881-3600', 'US', 'TX', 'HOUSTON', '77001', 'P O BOX 844', '', 'US', 'DE', '741753147.0', 'ADAMS RESOURCES & ENERGY INC', '19920703.0', '2-ACC', '0', '1231.0', '10-K', '20191231', '2019.0', 'FY', '20200306', '2020-03-06 16:50:00.0', '0', '1', 'ae-20191231_htm.xml', '1', '', 'q1', '2020']


### Define the financial tags we care about

In this cell, I define the specific GAAP tags related to operating cash flow and profitability.  
These tags appear in the NUM file and help me filter the dataset down to only the rows that are relevant to my research question.

I also create a small helper function (safe_float) that safely converts strings into floats so that bad values do not crash the notebook.

In [50]:
ocf_tags = [
    "NetCashProvidedByUsedInOperatingActivities",
    "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations",
]

profit_tags = [
    "NetIncomeLoss",
    "OperatingIncomeLoss",
]

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return None

### Filter the subset for operating cashflow and profitability rows

Here I:
1. Filter the subset NUM data to keep only rows where the `tag` is one of the operating cashflow or profitability tags.  
2. Extract the numeric value from the appropriate column.  
3. Quickly preview a few rows to make sure the filtering looks reasonable.

In [51]:
filtered = num_subset_split.filter(
    lambda cols: cols[1] in ocf_tags or cols[1] in profit_tags
)

tag_values = (
    filtered
    .map(lambda cols: (cols[1], safe_float(cols[8])))
    .filter(lambda x: x[1] is not None)
)

print("Example filtered rows from subset:")
for row in tag_values.take(10):
    print(row)

Example filtered rows from subset:
('OperatingIncomeLoss', 53369000.0)
('NetIncomeLoss', 11004241.0)
('NetCashProvidedByUsedInOperatingActivities', 19000000.0)
('NetIncomeLoss', 50577000.0)
('OperatingIncomeLoss', 378200000.0)
('NetIncomeLoss', 3209000.0)
('OperatingIncomeLoss', 253000.0)
('NetIncomeLoss', -96071000.0)
('NetIncomeLoss', -38929000.0)
('NetCashProvidedByUsedInOperatingActivities', 38191000.0)


### Summarize the subset to get average financial values
1. Map each tag to a `(value, 1)` pair  
2. Reduce by key to get the total sum and count for each tag  
3. Compute the average value for each financial metric   

In [52]:
mapped = tag_values.map(lambda x: (x[0], (x[1], 1)))
reduced = mapped.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
averages = reduced.map(lambda x: (x[0], x[1][0] / x[1][1]))

print("Average values for operating cash flow and profitability tags (subset):")
for tag, avg_val in averages.collect():
    print(tag, avg_val)

Average values for operating cash flow and profitability tags (subset):
NetIncomeLoss 75108943.67368421
OperatingIncomeLoss 185657975.71153846
NetCashProvidedByUsedInOperatingActivitiesContinuingOperations -19025.0
NetCashProvidedByUsedInOperatingActivities 338335191.0625


### Prepare the full NUM and SUB datasets for the full analysis
In this cell, I remove the header rows from the full NUM and SUB files and split each line by commas.  
This prepares the full dataset so I can build a complete analysis that addresses my research question
using all available company-year observations.

In [53]:
num_data = num_rdd.filter(lambda line: line != num_header)
sub_data = sub_rdd.filter(lambda line: line != sub_header)

num_split = num_data.map(lambda line: line.split(","))
sub_split = sub_data.map(lambda line: line.split(","))

print("Full NUM example row:", num_split.first())
print("Full SUB example row:", sub_split.first())

Full NUM example row: ['0001564590-20-010652', 'AccountsPayableCurrentAndNoncurrent', 'us-gaap/2019', '20181231', '0', 'USD', '', '', '607000.0', '', 'q1', '2020']
Full SUB example row: ['0000002178-20-000013', '2178', '"ADAMS RESOURCES & ENERGY', ' INC."', '5172.0', 'US', 'TX', 'HOUSTON', '77027', '17 S. BRIAR HOLLOW LN.', '', '713-881-3600', 'US', 'TX', 'HOUSTON', '77001', 'P O BOX 844', '', 'US', 'DE', '741753147.0', 'ADAMS RESOURCES & ENERGY INC', '19920703.0', '2-ACC', '0', '1231.0', '10-K', '20191231', '2019.0', 'FY', '20200306', '2020-03-06 16:50:00.0', '0', '1', 'ae-20191231_htm.xml', '1', '', 'q1', '2020']


### Extract operating cash flow and profitability rows from the full dataset

For each row, I pull out:
- the filing ID (adsh)
- the fiscal year (derived from the date)
- the tag (which tells me whether it is operating cashflow or profitability)
- the numeric value

I then filter to keep only the operating cashflow and profitability tags so I can build firm-year metrics.

In [54]:
num_filtered_full = num_split.filter(
    lambda cols: cols[1] in ocf_tags or cols[1] in profit_tags
)

num_tag_year_full = (
    num_filtered_full
    .map(
        lambda cols: (
            (cols[0], cols[3][:4], cols[1]),  
            safe_float(cols[8]),
        )
    )
    .filter(lambda x: x[1] is not None)
)

print("Sample (adsh, year, tag, value) rows from full NUM:")
for row in num_tag_year_full.take(10):
    print(row)

Sample (adsh, year, tag, value) rows from full NUM:
(('0001624794-20-000019', '2018', 'OperatingIncomeLoss'), 53369000.0)
(('0001239819-20-000032', '2018', 'NetIncomeLoss'), 11004241.0)
(('0001524358-20-000011', '2017', 'NetCashProvidedByUsedInOperatingActivities'), 19000000.0)
(('0001628280-20-001900', '2019', 'NetIncomeLoss'), 50577000.0)
(('0001564590-20-005754', '2018', 'OperatingIncomeLoss'), 378200000.0)
(('0001572758-20-000018', '2018', 'NetIncomeLoss'), 3209000.0)
(('0001628280-20-002318', '2018', 'OperatingIncomeLoss'), 253000.0)
(('0001102993-20-000017', '2019', 'NetIncomeLoss'), -96071000.0)
(('0001564590-20-007402', '2017', 'NetIncomeLoss'), -38929000.0)
(('0001297184-20-000007', '2018', 'NetCashProvidedByUsedInOperatingActivities'), 38191000.0)


### Build firm-year operating cashflow and profitability metrics

1. Separate the NUM rows into operating cashflow vs. profitability using the tag lists.  
2. For each group, aggregate values by (adsh, year) to get firm-year totals.  
3. Join the two RDDs so that each firm-year has both operating cashflow and profitability side by side.

In [55]:
ocf_records_full = num_tag_year_full.filter(lambda x: x[0][2] in ocf_tags)
profit_records_full = num_tag_year_full.filter(lambda x: x[0][2] in profit_tags)

ocf_firm_year_full = (
    ocf_records_full
    .map(lambda x: ((x[0][0], x[0][1]), x[1]))
    .reduceByKey(lambda a, b: a + b)
)

profit_firm_year_full = (
    profit_records_full
    .map(lambda x: ((x[0][0], x[0][1]), x[1]))
    .reduceByKey(lambda a, b: a + b)
)

print("Sample firm-year OCF:", ocf_firm_year_full.take(5))
print("Sample firm-year Profit:", profit_firm_year_full.take(5))

firm_year_metrics = ocf_firm_year_full.join(profit_firm_year_full)

print("Sample firm-year OCF + Profit rows:")
for row in firm_year_metrics.take(10):
    print(row)

Sample firm-year OCF: [(('0001141807-20-000005', '2019'), 15875000.0), (('0001171843-20-001695', '2017'), 17469000.0), (('0000320335-20-000008', '2017'), 3203624000.0), (('0000857855-20-000011', '2018'), 429020000.0), (('0000907254-20-000015', '2017'), 103450000.0)]


Sample firm-year Profit: [(('0001628280-20-002318', '2018'), 2245516000.0), (('0001534504-20-000030', '2019'), 5653000000.0), (('0001564590-20-005133', '2017'), 5570576000.0), (('0000936468-20-000016', '2018'), 46606000000.0), (('0001628280-20-001590', '2017'), -242262000.0)]
Sample firm-year OCF + Profit rows:
(('0001141807-20-000005', '2019'), (15875000.0, 54536000.0))
(('0001171843-20-001695', '2017'), (17469000.0, 30113000.0))
(('0000320335-20-000008', '2017'), (3203624000.0, 2938324000.0))
(('0000857855-20-000011', '2018'), (429020000.0, 498333000.0))
(('0000907254-20-000015', '2017'), (103450000.0, 48257000.0))
(('0000764180-20-000018', '2019'), (15674000000.0, 27343000000.0))
(('0001206942-20-000010', '2019'), (4530343.0, -2771933.0))
(('0001213900-20-005987', '2019'), (-23640.0, -143725.0))
(('0001213900-20-003785', '2019'), (780000.0, 1460000.0))
(('0001140361-20-007252', '2018'), (-3139154.0, -19727741.0))


### Join operating cashflow and profitability per firm-year
- a total operating cash flow value, and  
- a total profitability value.

In [56]:
firm_year_metrics = ocf_firm_year_full.join(profit_firm_year_full)
print("Sample firm-year OCF + Profit rows:")
for row in firm_year_metrics.take(10):
    print(row)

Sample firm-year OCF + Profit rows:
(('0001141807-20-000005', '2019'), (15875000.0, 54536000.0))
(('0001171843-20-001695', '2017'), (17469000.0, 30113000.0))
(('0000320335-20-000008', '2017'), (3203624000.0, 2938324000.0))
(('0000857855-20-000011', '2018'), (429020000.0, 498333000.0))
(('0000907254-20-000015', '2017'), (103450000.0, 48257000.0))
(('0000764180-20-000018', '2019'), (15674000000.0, 27343000000.0))
(('0001206942-20-000010', '2019'), (4530343.0, -2771933.0))
(('0001213900-20-005987', '2019'), (-23640.0, -143725.0))
(('0001213900-20-003785', '2019'), (780000.0, 1460000.0))
(('0001140361-20-007252', '2018'), (-3139154.0, -19727741.0))


### Attach SIC industry codes from SUB
- extract (adsh, sic) pairs from SUB,  
- re-key the firm-year metrics by adsh, and  
- join them together.

This uses an RDD join on adsh instead of any pre-merged CSV file, which makes the
analysis portable to Dataproc because it only relies on the original SEC files in GCS.

In [57]:
adsh_sic = (
    sub_split
    .map(lambda cols: (cols[0], cols[3]))  
    .filter(lambda x: x[1] not in ("", None, "NA"))
)

print("Sample (adsh, sic) pairs:", adsh_sic.take(5))

firm_year_by_adsh = firm_year_metrics.map(
    lambda x: (x[0][0], (x[0][1], x[1][0], x[1][1])) 
)

joined_full = firm_year_by_adsh.join(adsh_sic)

print("Sample rows with industry:", joined_full.take(10))

Sample (adsh, sic) pairs: [('0000002178-20-000013', ' INC."'), ('0000002488-20-000008', '3674.0'), ('0000002969-20-000010', '2810.0'), ('0000003499-20-000005', '6798.0'), ('0000003545-20-000039', ' INC."')]


Sample rows with industry: [('0001609471-20-000004', (('2017', 49720000.0, 25843000.0), '6798.0')), ('0001609471-20-000004', (('2018', 68956000.0, 25182000.0), '6798.0')), ('0001609471-20-000004', (('2019', 63502000.0, 22331000.0), '6798.0')), ('0001564590-20-006373', (('2017', 122389000.0, 497320000.0), '7359.0')), ('0001564590-20-006373', (('2019', 187994000.0, 714534000.0), '7359.0')), ('0001564590-20-006373', (('2018', 142667000.0, 590661000.0), '7359.0')), ('0001437749-20-004678', (('2019', 3101000.0, 27383000.0), '3842.0')), ('0001437749-20-004678', (('2018', 2000000.0, 31970000.0), '3842.0')), ('0000939057-20-000046', (('2019', 17674000.0, 16982000.0), '6035.0')), ('0000939057-20-000046', (('2018', 20663000.0, 17447000.0), '6035.0'))]


### Compute average operating cashflow and profitability by industry-year

For each (sic, year) combination:
- sum all operating cashflow values  
- sum all profitability values  
- count how many firm-years are included  
- compute the average operating cashflow and average profitability for that industry and year.

This directly answers my question about how these two metrics move together across different industries.


In [ ]:
industry_year_pairs = joined_full.map(
    lambda x: (
        (x[1][1], x[1][0][0]),          
        (x[1][0][1], x[1][0][2], 1),    
    )
)

industry_year_agg = industry_year_pairs.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1], a[2] + b[2])
)

industry_year_avg = industry_year_agg.map(
    lambda x: (
        x[0],  # (sic, year)
        (x[1][0] / x[1][2], x[1][1] / x[1][2]), 
    )
)

industry_year_avg_sorted = industry_year_avg.sortBy(
    lambda x: (x[0][1], x[0][0])  
)

In [59]:
print("Industry-Year Average Operating Cash Flow & Profitability:\n")
for (key, vals) in industry_year_avg_sorted.take(50):
    sic, year = key
    avg_ocf, avg_profit = vals
    print(f"SIC {sic}, Year {year} Avg OCF = {avg_ocf:.2f}, Avg Profit = {avg_profit:.2f}")

Industry-Year Average Operating Cash Flow & Profitability:

SIC 3674.0, Year 2010 Avg OCF = -333000.00, Avg Profit = 35947000.00
SIC 3674.0, Year 2011 Avg OCF = -28333.33, Avg Profit = -2835000.00
SIC  INC.", Year 2014 Avg OCF = -117405.00, Avg Profit = -511539.00
SIC 7381.0, Year 2014 Avg OCF = -46937.00, Avg Profit = -71270.00
SIC 7389.0, Year 2014 Avg OCF = 63.00, Avg Profit = -282018.00
SIC  INC.", Year 2015 Avg OCF = -359555.20, Avg Profit = -2903118.40
SIC 6798.0, Year 2015 Avg OCF = 2888000.00, Avg Profit = 24000.00
SIC 7381.0, Year 2015 Avg OCF = 1201.40, Avg Profit = -610589.60
SIC 7389.0, Year 2015 Avg OCF = -103937.00, Avg Profit = -565366.00
SIC  INC.", Year 2016 Avg OCF = 34359118.72, Avg Profit = 18186781.78
SIC 1000.0, Year 2016 Avg OCF = -288672.00, Avg Profit = -1176472.00
SIC 2834.0, Year 2016 Avg OCF = -15486000.00, Avg Profit = -47910000.00
SIC 2836.0, Year 2016 Avg OCF = -52861000.00, Avg Profit = -173650000.00
SIC 3089.0, Year 2016 Avg OCF = -76886.00, Avg Profit 

### Final Answer

My question was whether companies with strong operating cashflow also show higher profitability, and how this looks across different industries.

After building firm-year totals for operating cashflow and profitability, joining them with SIC industry codes, and averaging by (industry, year), the results mostly show a positive relationship industries and years with higher operating cashflow generally also have higher profitability.

There are a few exceptions where operating cashflow is positive but profitability is low or negative, which appear to be outliers or industry-specific situations. Overall, the trend is there, but the strength of the relationship changes depending on the industry.